In [8]:
from sklearn.feature_selection import SelectKBest, chi2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from google.colab import files
import io
from imblearn.under_sampling import RandomUnderSampler

In [17]:
uploaded = files.upload()

# Assuming only one file is uploaded and it's a CSV
df = None
if uploaded:
    filename = list(uploaded.keys())[0]
    uploaded_file_content = uploaded[filename]
    print(f'User uploaded file "{filename}" with length {len(uploaded_file_content)} bytes')

    # Read the byte content into a pandas DataFrame, assuming CSV format
    try:
        df = pd.read_csv(io.BytesIO(uploaded_file_content))
        print(f'Successfully loaded "{filename}" into a pandas DataFrame.')
        display(df.head()) # Display the first 5 rows to confirm
    except Exception as e:
        print(f'Error reading file into DataFrame: {e}')
else:
    print('No file was uploaded.')

Saving processed_data_IT25103141.csv to processed_data_IT25103141 (1).csv
User uploaded file "processed_data_IT25103141 (1).csv" with length 18190568 bytes
Successfully loaded "processed_data_IT25103141 (1).csv" into a pandas DataFrame.


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,0.0,0.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,1.0,...,0.0,2.0,0.0,0.0,0.0,1.0,2.0,4.0,5.0,0.0
1,1.0,0.0,1.0,23.0,1.0,0.0,0.0,1.0,1.0,1.0,...,0.0,2.0,0.0,0.0,0.0,1.0,13.0,4.0,7.0,0.0
2,1.0,1.0,1.0,29.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,9.0,6.0,8.0,0.0
3,1.0,1.0,1.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,0.0,0.0,1.0,7.0,4.0,7.0,0.0
4,0.0,1.0,1.0,16.0,1.0,0.0,0.0,1.0,1.0,1.0,...,1.0,5.0,30.0,30.0,1.0,0.0,7.0,5.0,1.0,0.0


### Separate Features and Target for Undersampling

Given that the uploaded `df` is already processed, we'll now explicitly separate the features (`pdx_final_df`) and the target variable (`pduse`). Based on the columns in your `df`, I'm assuming `HeartDiseaseorAttack_1.0` is your target variable. We will use this to create the `pduse` and `pdx_final_df` variables needed for undersampling.

In [15]:
print(df.columns)

Index(['HighBP', 'HighChol', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack',
       'PhysActivity', 'HvyAlcoholConsump', 'GenHlth', 'MentHlth', 'PhysHlth',
       'DiffWalk', 'Age', 'Education', 'Income'],
      dtype='object')


In [18]:
# Define the target column for undersampling
target_column = 'Diabetes_binary'

# Extract the target variable (pduse)
pduse = df[target_column]

# Define the features (pdx_final_df) by dropping the target column(s) from the DataFrame
# We drop both HeartDiseaseorAttack_0.0 and HeartDiseaseorAttack_1.0 as they represent the same original feature.
pdx_final_df = df.drop(columns=['HeartDiseaseorAttack_0.0', 'HeartDiseaseorAttack_1.0'], errors='ignore')

print(f"Shape of features (pdx_final_df): {pdx_final_df.shape}")
print(f"Shape of target (pduse): {pduse.shape}")

print("\nFirst 5 rows of features (pdx_final_df):")
display(pdx_final_df.head())

print("\nFirst 5 rows of target (pduse):")
display(pduse.head())

Shape of features (pdx_final_df): (202944, 22)
Shape of target (pduse): (202944,)

First 5 rows of features (pdx_final_df):


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,0.0,0.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,1.0,...,0.0,2.0,0.0,0.0,0.0,1.0,2.0,4.0,5.0,0.0
1,1.0,0.0,1.0,23.0,1.0,0.0,0.0,1.0,1.0,1.0,...,0.0,2.0,0.0,0.0,0.0,1.0,13.0,4.0,7.0,0.0
2,1.0,1.0,1.0,29.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,9.0,6.0,8.0,0.0
3,1.0,1.0,1.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,0.0,0.0,1.0,7.0,4.0,7.0,0.0
4,0.0,1.0,1.0,16.0,1.0,0.0,0.0,1.0,1.0,1.0,...,1.0,5.0,30.0,30.0,1.0,0.0,7.0,5.0,1.0,0.0



First 5 rows of target (pduse):


,Diabetes_binary
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0


### Apply Random UnderSampler

Now we will apply `RandomUnderSampler` to balance the class distribution of your target variable. This will create a new dataset where the number of instances in each class of the target variable is more balanced.

In [19]:
# random_state for reproducibility
rus = RandomUnderSampler(random_state=42)

print(f"Shape of features before undersampling: {pdx_final_df.shape}")
print(f"Shape of target before undersampling: {pduse.shape}")
print(f"Class distribution before undersampling:\n{pduse.value_counts()}")

# Apply undersampling to the processed features and original target variable
X_resampled, y_resampled = rus.fit_resample(pdx_final_df, pduse)

print(f"\nFeatures - after undersampling: {X_resampled.shape}")
print(f"Target - after undersampling: {y_resampled.shape}")
print(f"Class distribution after undersampling:\n{y_resampled.value_counts()}")

# Display the first few rows of the resampled features
print("\nFirst 5 rows of resampled features:")
display(X_resampled.head())

Shape of features before undersampling: (202944, 22)
Shape of target before undersampling: (202944,)
Class distribution before undersampling:
Diabetes_binary
0.0    174667
1.0     28277
Name: count, dtype: int64

Features - after undersampling: (56554, 22)
Target - after undersampling: (56554,)
Class distribution after undersampling:
Diabetes_binary
0.0    28277
1.0    28277
Name: count, dtype: int64

First 5 rows of resampled features:


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
75362,1.0,0.0,1.0,25.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,2.0,0.0,0.0,0.0,0.0,8.0,4.0,8.0,0.0
61144,1.0,1.0,1.0,31.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,3.0,0.0,0.0,0.0,0.0,12.0,4.0,6.0,0.0
143246,1.0,1.0,1.0,27.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,3.0,0.0,0.0,0.0,1.0,11.0,6.0,8.0,0.0
5798,0.0,0.0,1.0,23.0,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,3.0,30.0,0.0,0.0,1.0,10.0,5.0,6.0,0.0
95300,0.0,0.0,1.0,22.0,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,3.0,0.0,0.0,0.0,0.0,3.0,6.0,6.0,0.0


### Download Resampled Data

Finally, we will save the undersampled features (`X_resampled`) and target (`y_resampled`) to CSV files and provide them for download.

In [ ]:
# X_resampled
features_filename = 'undersampled_features.csv'
X_resampled.to_csv(features_filename, index=False)
print(f"'{features_filename}'")
files.download(features_filename)

# y_resampled
target_filename = 'undersampled_target.csv'
y_resampled.to_csv(target_filename, index=False)
print(f"'{target_filename}'")
files.download(target_filename)

'undersampled_features.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'undersampled_target.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
## test set preprocessing
## upload the test file here
print("upload the test set!")

uploaded = files.upload()
# Assuming only one file is uploaded and it's a CSV
df = None
if uploaded:
    filename = list(uploaded.keys())[0]
    uploaded_file_content = uploaded[filename]
    print(f'User uploaded file "{filename}" with length {len(uploaded_file_content)} bytes')

    # Read the byte content into a pandas DataFrame, assuming CSV format
    try:
        df = pd.read_csv(io.BytesIO(uploaded_file_content))
        print(f'Successfully loaded "{filename}" into a pandas DataFrame.')
        display(df.head()) # Display the first 5 rows to confirm
    except Exception as e:
        print(f'Error reading file into DataFrame: {e}')
else:
    print('No file was uploaded.')

upload the test set!


No file was uploaded.


In [ ]:
# test set preprocessing
# 1. Drop the same features from xtest
xtest_after_drop = xtest.drop(columns=features_to_drop, errors='ignore').copy()

# 2. Transform the test set using the preprocessor fitted on training data
xtest_processed = preprocessor.transform(xtest_after_drop)

# 3. Convert to DataFrame for consistency
xtest_final_df = pd.DataFrame(xtest_processed, columns=feature_names, index=xtest.index)

print(f"Original xtest shape: {xtest.shape}")
print(f"Processed xtest shape: {xtest_final_df.shape}")

print("\nFirst 5 rows of processed test features:")
display(xtest_final_df.head())

## downloading xtest (processed file)
from google.colab import files

# Save the preprocessed test features to a CSV file
test_features_filename = 'preprocessed_xtest.csv'
xtest_final_df.to_csv(test_features_filename, index=False)
print(f"Saved '{test_features_filename}'")

# Download the file
files.download(test_features_filename)

NameError: name 'xtest' is not defined